<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-07-blob-processing-classification-routing/notebooks/Pfizer_Step2_Query_Routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Route Queries Across Multiple PDF Files
**Pfizer AI Document Intelligence Externship — Step 2**

Metadata-first routing: classify the *question*, classify every *page*, then filter — all
before any embedding or full-text search runs.

Corpus: `pharma-blob-sample.pdf` (Cytiva) + `pharma-blob-test.pdf` (West Pharmaceutical).
Model: **Mistral-7B-Instruct-v0.3**, 4-bit on a Colab T4.

Pipeline: `classify_query_llm()` → `classify_doc_type_llm()` over `pdf_metadata_store` →
filter to `matched_documents` → keyword re-rank → routing dictionary.

> **Runtime → Change runtime type → T4 GPU** before running.

## 1. Environment

In [ ]:
!pip -q install "transformers>=4.44" accelerate bitsandbytes pymupdf gdown pandas
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA: True | Tesla T4


## 2. Load the document set

Two blob PDFs, each holding several distinct documents. One record per **page** — the page
is the unit a router can actually return.

In [ ]:
import os, gdown

FILES = [
    ("pharma-blob-sample.pdf", "1fJ7DszcjB_TerDTAhXfAw74VqJBElTyw"),
    ("pharma-blob-test.pdf",   "1atOj_CzvPRm-CskbhvJdzlJIpcEanVu1"),
]

for name, fid in FILES:
    if not os.path.exists(name):
        gdown.download(id=fid, output=name, quiet=True)
    print(name, os.path.getsize(name), "bytes")

pharma-blob-sample.pdf 12768 bytes
pharma-blob-test.pdf 16368 bytes


In [ ]:
import re, json, uuid
from collections import Counter
import pymupdf

YEAR_RE = re.compile(r"\b(?:19|20)\d{2}\b")
COMPACT_DATE_RE = re.compile(r"\b((?:19|20)\d{2})(?:0[1-9]|1[0-2])(?:0[1-9]|[12]\d|3[01])\b")

def guess_year(text):
    """Most frequent 4-digit year on the page; ties resolve to the earliest."""
    yrs = COMPACT_DATE_RE.findall(text) + YEAR_RE.findall(text)
    if not yrs:
        return "unknown"
    counts = Counter(yrs)
    top = max(counts.values())
    return min(y for y, c in counts.items() if c == top)


def build_store(files, user_id="xyz"):
    """One dict per page: the pdf_metadata_store the router filters over."""
    store = []
    for filename, _ in files:
        file_id = str(uuid.uuid5(uuid.NAMESPACE_URL, filename))
        doc = pymupdf.open(filename)
        for i, page in enumerate(doc):
            text = page.get_text("text").strip()
            store.append({
                "file_id": file_id,
                "user_id": user_id,
                "doc_type": None,          # filled in by classify_doc_type_llm()
                "year": guess_year(text),
                "filename": filename,
                "page_number": i,
                "text": text,
            })
    return store


def window(text, n=1200):
    text = re.sub(r"[ \t]+", " ", re.sub(r"\n{3,}", "\n\n", text)).strip()
    return text if len(text) <= n else text[:n] + "\n...[truncated]"


pdf_metadata_store = build_store(FILES)
print(len(pdf_metadata_store), "pages across", len(FILES), "files\n")
for p in pdf_metadata_store:
    first = p["text"].splitlines()[0] if p["text"].strip() else "(EMPTY)"
    print(f"{p['filename'][:24]:<24} p{p['page_number']:>2} | {p['year']:<8} | {first[:58]}")

20 pages across 2 files

pharma-blob-sample.pdf   p 0 | 2022     | Cytiva
pharma-blob-sample.pdf   p 1 | 2024     | Certificate of Quality
pharma-blob-sample.pdf   p 2 | 2019     | Certificate of Quality
pharma-blob-sample.pdf   p 3 | 2023     | Packaging Component Specification
pharma-blob-sample.pdf   p 4 | 2023     | Packaging Component Specification (continued)
pharma-blob-sample.pdf   p 5 | 2023     | Declaration Regarding Transmissible Spongiform
pharma-blob-sample.pdf   p 6 | unknown  | Material Description Sheet
pharma-blob-sample.pdf   p 7 | 2019     | Supplier Qualification Record
pharma-blob-sample.pdf   p 8 | 2019     | Supplier Qualification Record (continued)
pharma-blob-sample.pdf   p 9 | 2024     | Cytiva
pharma-blob-test.pdf     p 0 | 2024     | From:
pharma-blob-test.pdf     p 1 | 2016     | West Pharmaceutical Services, Inc.
pharma-blob-test.pdf     p 2 | 2024     | QUALITY CERTIFICATE No. 80372784-000010 (continued)
pharma-blob-test.pdf     p 3 | 2024     | QUALITY 

## 3. Load the open-source model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"   # ungated mirror, same weights

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto", torch_dtype=torch.float16)
model.eval()
print("loaded:", MODEL_ID)

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:271: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded: unsloth/mistral-7b-instruct-v0.3-bnb-4bit


In [ ]:
RAW_OUTPUTS = []   # every raw generation, kept so wrong answers can be inspected later

@torch.inference_mode()
def ask(messages, max_new_tokens=24):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, pad_token_id=tokenizer.eos_token_id)
    txt = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    RAW_OUTPUTS.append(txt)
    return txt

## 4. The label set and the two prompts

Both prompts share one closed label list and both must answer with a single JSON object.
The query prompt carries an explicit mapping from *what the user is asking about* to the
document that holds it — that mapping is what does the routing work.

In [ ]:
DOC_TYPES = ["cover_letter", "certificate_of_quality", "packaging_specification",
             "bse_tse_declaration", "material_description", "supplier_qualification",
             "chain_of_custody", "unknown"]

QUERY_RULES = f"""You route user questions to the right kind of pharmaceutical supplier document.
Read the QUESTION and answer with the one document type most likely to contain the answer.
Choose EXACTLY one label from this list:
{chr(10).join('- ' + t for t in DOC_TYPES)}

Guidance:
- lot or batch numbers, test results, release statements -> certificate_of_quality
- a part or article number tied to a released product or lot -> certificate_of_quality
- a part number for a packaging component (tray, film, carton) -> packaging_specification
- blister trays, cartons, lid film, drawings, ECN revisions -> packaging_specification
- animal-derived material, bovine or porcine origin, BSE or TSE risk -> bse_tse_declaration
- materials of construction, resins, sterilization compatibility, dimensions -> material_description
- audits, ISO certificates, approved suppliers, sub-suppliers, delivery performance -> supplier_qualification
- where a product was made, shipping route, traceability of a lot -> chain_of_custody
- who sent the packet, what documents it contains -> cover_letter
- a question too vague to place with confidence -> unknown

Answer with JSON only, no prose: {{"doc_type": "<label>"}}"""

PAGE_RULES = f"""You classify one page of a pharmaceutical supplier document packet.
Read the PAGE and answer with the document type it belongs to.
Choose EXACTLY one label from this list:
{chr(10).join('- ' + t for t in DOC_TYPES)}

Notes:
- a covering email or transmittal letter listing the enclosed documents -> cover_letter
- "Certificate of Quality" / "Quality Certificate" with lots and test results -> certificate_of_quality
- an animal origin or BSE/TSE statement -> bse_tse_declaration
- a continuation page keeps the type of the document it continues
- if the page does not match any label, answer unknown

Answer with JSON only, no prose: {{"doc_type": "<label>"}}"""


def query_messages(query):
    return [
        {"role": "user", "content": QUERY_RULES + "\n\nQUESTION: Does this product contain any bovine derived material?"},
        {"role": "assistant", "content": '{"doc_type": "bse_tse_declaration"}'},
        {"role": "user", "content": "QUESTION: What lot numbers were released and did they pass the flow rate test?"},
        {"role": "assistant", "content": '{"doc_type": "certificate_of_quality"}'},
        {"role": "user", "content": "QUESTION: When was the supplier last audited?"},
        {"role": "assistant", "content": '{"doc_type": "supplier_qualification"}'},
        {"role": "user", "content": f"QUESTION: {query}"},
    ]


def page_messages(page_text):
    return [
        {"role": "user", "content": PAGE_RULES + "\n\nPAGE:\nCertificate of Quality\nIssued by Cytiva Westborough Quality Assurance\nProduct: AKTA ready Low Flow Kit\nLot Number: 18356721\nProduct Release Criteria..."},
        {"role": "assistant", "content": '{"doc_type": "certificate_of_quality"}'},
        {"role": "user", "content": "PAGE:\nDeclaration Regarding Transmissible Spongiform Encephalopathies (BSE/TSE Compliance Statement)\nWe hereby declare that the products do not contain materials of animal origin..."},
        {"role": "assistant", "content": '{"doc_type": "bse_tse_declaration"}'},
        {"role": "user", "content": f"PAGE:\n{window(page_text)}"},
    ]

print(QUERY_RULES)

You route user questions to the right kind of pharmaceutical supplier document.
Read the QUESTION and answer with the one document type most likely to contain the answer.
Choose EXACTLY one label from this list:
- cover_letter
- certificate_of_quality
- packaging_specification
- bse_tse_declaration
- material_description
- supplier_qualification
- chain_of_custody
- unknown

Guidance:
- lot or batch numbers, test results, release statements -> certificate_of_quality
- a part or article number tied to a released product or lot -> certificate_of_quality
- a part number for a packaging component (tray, film, carton) -> packaging_specification
- blister trays, cartons, lid film, drawings, ECN revisions -> packaging_specification
- animal-derived material, bovine or porcine origin, BSE or TSE risk -> bse_tse_declaration
- materials of construction, resins, sterilization compatibility, dimensions -> material_description
- audits, ISO certificates, approved suppliers, sub-suppliers, delivery 

## 5. Output repair and deterministic fallbacks

`parse_label` accepts anything the model emits — fenced JSON, prose, a bare label — and
maps it onto the closed list. If nothing survives, a cue-based heuristic answers instead,
so a routing decision is always produced.

In [ ]:
_ALIASES = {
    "certificate of quality": "certificate_of_quality",
    "quality certificate": "certificate_of_quality",
    "certificate of analysis": "certificate_of_quality",
    "coa": "certificate_of_quality",
    "packaging component specification": "packaging_specification",
    "packaging spec": "packaging_specification",
    "bse/tse": "bse_tse_declaration",
    "animal origin": "bse_tse_declaration",
    "declaration": "bse_tse_declaration",
    "material description sheet": "material_description",
    "material description": "material_description",
    "compliance bulletin": "material_description",
    "supplier qualification record": "supplier_qualification",
    "supplier qualification": "supplier_qualification",
    "chain of custody": "chain_of_custody",
    "certificate of processing": "chain_of_custody",
    "cover letter": "cover_letter",
    "letter": "cover_letter",
}

def normalize_type(raw):
    """Map a free-text label onto the closed list; never raises."""
    if not raw:
        return None
    s = str(raw).strip().strip('".').lower().replace("-", " ")
    if s.replace(" ", "_") in DOC_TYPES:
        return s.replace(" ", "_")
    if s in _ALIASES:
        return _ALIASES[s]
    for k, v in _ALIASES.items():
        if k in s:
            return v
    for t in DOC_TYPES:
        if t.replace("_", " ") in s:
            return t
    return None


def parse_label(raw):
    """Strict JSON -> first {...} block -> label scrape. Never raises."""
    if not raw:
        return None
    txt = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    m = re.search(r"\{[^{}]*\}", txt, re.S)
    for cand in [txt, m.group(0) if m else None]:
        if not cand:
            continue
        try:
            obj = json.loads(cand)
        except Exception:
            continue
        if isinstance(obj, dict):
            hit = normalize_type(obj.get("doc_type") or obj.get("type") or obj.get("label"))
            if hit:
                return hit
    low = txt.lower().replace("-", " ")
    for t in DOC_TYPES:
        if t in low or t.replace("_", " ") in low:
            return t
    for k, v in _ALIASES.items():
        if k in low:
            return v
    return None

In [ ]:
HEADING_CUES = {
    "certificate of quality": "certificate_of_quality",
    "quality certificate": "certificate_of_quality",
    "packaging component specification": "packaging_specification",
    "transmissible spongiform": "bse_tse_declaration",
    "bse/tse": "bse_tse_declaration",
    "animal origin statement": "bse_tse_declaration",
    "material description sheet": "material_description",
    "compliance bulletin": "material_description",
    "supplier qualification record": "supplier_qualification",
    "chain of custody": "chain_of_custody",
    "certificate of processing": "chain_of_custody",
    "to whom it may concern": "cover_letter",
    "dear ": "cover_letter",
    "subject:": "cover_letter",
}

def heuristic_doc_type(text):
    head = "\n".join(text.strip().splitlines()[:12]).lower()   # letterheads push the real title down
    for k, v in HEADING_CUES.items():
        if k in head:
            return v
    low = text.lower()
    for k, v in HEADING_CUES.items():
        if k in low:
            return v
    return "unknown"


QUERY_CUES = {
    "certificate_of_quality": ["lot number", "batch", "test result", "released", "hardness",
                               "conform", "certificate", "article number", "part number",
                               "expiration", "shore", "specification limit", "coa"],
    "packaging_specification": ["packaging", "blister", "carton", "tray", "lid film",
                                "ecn", "revision", "drawing", "primary packaging"],
    "bse_tse_declaration": ["bse", "tse", "animal origin", "animal derived", "animal",
                            "bovine", "spongiform", "porcine", "ovine"],
    "material_description": ["material of construction", "resin", "polypropylene", "silicone",
                             "sterilization", "autoclave", "gamma irradiation", "compatib",
                             "physical properties"],
    "supplier_qualification": ["supplier", "audit", "iso 9001", "iso 13485", "qualification",
                               "sub supplier", "on time delivery", "approved vendor"],
    "chain_of_custody": ["chain of custody", "traceab", "traced", "shipped", "custody",
                         "distribution centre", "distribution center", "manufacturing site",
                         "where was it made", "which facility"],
    "cover_letter": ["cover letter", "who sent", "transmittal", "which documents are included",
                     "what is in this package", "regards"],
}

def heuristic_query_type(query):
    """Weighted cue match - a two-word cue outranks a one-word cue that also fires."""
    q = " " + re.sub(r"[^a-z0-9 ]", " ", query.lower()) + " "
    best, score = "unknown", 0
    for t, cues in QUERY_CUES.items():
        s = sum(len(c.split()) ** 2 for c in cues if c in q)
        if s > score:
            best, score = t, s
    return best

## 6. The two LLM classifiers

In [ ]:
def classify_query_llm(query):
    """User question -> predicted_doc_type (one label from DOC_TYPES)."""
    raw = ""
    try:
        raw = ask(query_messages(query))
    except Exception as e:
        print(f"  [warn] query classification failed ({type(e).__name__}) - using heuristic")
    return parse_label(raw) or heuristic_query_type(query)


def classify_doc_type_llm(page_text):
    """One page of text -> doc_type (one label from DOC_TYPES)."""
    raw = ""
    try:
        raw = ask(page_messages(page_text))
    except Exception as e:
        print(f"  [warn] page classification failed ({type(e).__name__}) - using heuristic")
    return parse_label(raw) or heuristic_doc_type(page_text)


CONTINUATION_CUES = ("(continued)", "cont'd", "page 2 of", "page 3 of")

def reconcile_page_type(page_text, llm_label, prev_label=None):
    """Two deterministic overrides on top of the model's answer:
    a '(continued)' page inherits the previous page's type, and an explicit
    document title in the header beats a model that disagrees with it."""
    head = "\n".join(page_text.strip().splitlines()[:12]).lower()
    if any(c in head for c in CONTINUATION_CUES) and prev_label:
        return prev_label, "continuation"
    cue = next((v for k, v in HEADING_CUES.items() if k in head), None)
    if cue and cue != llm_label:
        return cue, "header_override"
    return (llm_label or heuristic_doc_type(page_text)), ""


print(classify_query_llm("What are the part numbers listed in this document?"))

[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


packaging_specification


## 7. Assign a doc_type to every page in the store

In [ ]:
import time
t0 = time.time()

prev_label, prev_file = None, None
for p in pdf_metadata_store:
    if p["filename"] != prev_file:          # never inherit across a file boundary
        prev_label, prev_file = None, p["filename"]
    p["llm_label"] = classify_doc_type_llm(p["text"])
    p["llm_raw"] = RAW_OUTPUTS[-1]
    p["doc_type"], p["override"] = reconcile_page_type(p["text"], p["llm_label"], prev_label)
    prev_label = p["doc_type"]

corrected = [p for p in pdf_metadata_store if p["override"]]
print(f"labelled {len(pdf_metadata_store)} pages in {time.time()-t0:.1f}s")
print(f"{len(corrected)} page(s) corrected by a deterministic override:")
for p in corrected:
    print(f"  {p['filename']} p{p['page_number']}: {p['llm_label']} -> {p['doc_type']}  ({p['override']})")

[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

labelled 20 pages in 130.2s
8 page(s) corrected by a deterministic override:
  pharma-blob-sample.pdf p0: packaging_specification -> cover_letter  (header_override)
  pharma-blob-sample.pdf p4: packaging_specification -> packaging_specification  (continuation)
  pharma-blob-sample.pdf p8: supplier_qualification -> supplier_qualification  (continuation)
  pharma-blob-test.pdf p2: packaging_specification -> certificate_of_quality  (continuation)
  pharma-blob-test.pdf p3: certificate_of_quality -> certificate_of_quality  (continuation)
  pharma-blob-test.pdf p6: packaging_specification -> material_description  (header_override)
  pharma-blob-test.pdf p8: supplier_qualification -> supplier_qualification  (continuation)
  pharma-blob-test.pdf p9: packaging_specification -> chain_of_custody  (header_override)


In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 46)

store_df = pd.DataFrame(pdf_metadata_store)
store_df["first_line"] = store_df["text"].str.split("\n").str[0].str[:40]
store_df[["filename", "page_number", "llm_label", "doc_type", "override", "year", "first_line"]]

,filename,page_number,llm_label,doc_type,override,year,first_line
0,pharma-blob-sample.pdf,0,packaging_specification,cover_letter,header_override,2022,Cytiva
1,pharma-blob-sample.pdf,1,certificate_of_quality,certificate_of_quality,,2024,Certificate of Quality
2,pharma-blob-sample.pdf,2,certificate_of_quality,certificate_of_quality,,2019,Certificate of Quality
3,pharma-blob-sample.pdf,3,packaging_specification,packaging_specification,,2023,Packaging Component Specification
4,pharma-blob-sample.pdf,4,packaging_specification,packaging_specification,continuation,2023,Packaging Component Specification (conti
5,pharma-blob-sample.pdf,5,bse_tse_declaration,bse_tse_declaration,,2023,Declaration Regarding Transmissible Spon
6,pharma-blob-sample.pdf,6,material_description,material_description,,unknown,Material Description Sheet
7,pharma-blob-sample.pdf,7,supplier_qualification,supplier_qualification,,2019,Supplier Qualification Record
8,pharma-blob-sample.pdf,8,supplier_qualification,supplier_qualification,continuation,2019,Supplier Qualification Record (continued
9,pharma-blob-sample.pdf,9,chain_of_custody,chain_of_custody,,2024,Cytiva


In [ ]:
print(store_df["doc_type"].value_counts().to_string())
print()
print(store_df[["filename", "page_number", "doc_type", "year", "first_line"]].to_string(index=False))

doc_type
certificate_of_quality     5
supplier_qualification     4
packaging_specification    3
cover_letter               2
bse_tse_declaration        2
material_description       2
chain_of_custody           2

              filename  page_number                doc_type    year                               first_line
pharma-blob-sample.pdf            0            cover_letter    2022                                   Cytiva
pharma-blob-sample.pdf            1  certificate_of_quality    2024                   Certificate of Quality
pharma-blob-sample.pdf            2  certificate_of_quality    2019                   Certificate of Quality
pharma-blob-sample.pdf            3 packaging_specification    2023        Packaging Component Specification
pharma-blob-sample.pdf            4 packaging_specification    2023 Packaging Component Specification (conti
pharma-blob-sample.pdf            5     bse_tse_declaration    2023 Declaration Regarding Transmissible Spon
pharma-blob-sample.pdf  

## 8. Filter, then keyword fallback

`matched_documents` is the metadata filter. The keyword score only *re-ranks inside* that
filtered set — and widens to the whole store when the predicted type matches nothing,
which is what stops a vague question from returning an empty answer.

In [ ]:
# every page now carries a label from the closed list - the precondition for filtering
assert all(p["doc_type"] in DOC_TYPES for p in pdf_metadata_store)
kinds = sorted({p["doc_type"] for p in pdf_metadata_store})
print(len(pdf_metadata_store), "pages |", len(kinds), "distinct doc_types:", ", ".join(kinds))

20 pages | 7 distinct doc_types: bse_tse_declaration, certificate_of_quality, chain_of_custody, cover_letter, material_description, packaging_specification, supplier_qualification


In [ ]:
STOP = set("what is are the a an of in for this that which do does list show me tell "
           "all any there was were on to and or with how many number numbers".split())

FALLBACK_TERMS = ["part number", "article number", "specification", "certificate",
                  "lot", "batch", "sterilization", "bse", "supplier", "custody"]

def keyword_score(query, text):
    low = text.lower()
    terms = [w for w in re.findall(r"[a-z0-9]+", query.lower()) if w not in STOP and len(w) > 2]
    score = sum(low.count(w) for w in terms)
    score += 2 * sum(1 for t in FALLBACK_TERMS if t in query.lower() and t in low)
    return score


def route_query(query, store=None, top_k=3, preview=200):
    store = pdf_metadata_store if store is None else store
    predicted = classify_query_llm(query) or "unknown"
    matched = [p for p in store if p["doc_type"] == predicted]
    widened = False
    if not matched:                       # nothing of that type -> keyword search everything
        widened, matched = True, list(store)
    scored = [(keyword_score(query, p["text"]), p) for p in matched]
    scored.sort(key=lambda x: -x[0])
    hits = [sp for sp in scored if sp[0] > 0] or scored
    out = []
    for s, p in hits[:top_k]:
        r = {k: v for k, v in p.items() if k not in ("llm_raw", "llm_label", "override")}
        r["match_score"] = s
        r["text"] = re.sub(r"\s+", " ", p["text"])[:preview] + "..."
        out.append(r)
    return {"query": query,
            "predicted_doc_type": predicted,
            "widened_to_full_store": widened,
            "matched_documents": out}

## 9. Route a query

In [ ]:
result = route_query("What are the part numbers listed in this document?")
print(json.dumps(result, indent=2)[:2200])

[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "query": "What are the part numbers listed in this document?",
  "predicted_doc_type": "packaging_specification",
  "widened_to_full_store": false,
  "matched_documents": [
    {
      "file_id": "1c89f2db-a339-5d5e-92b1-dc0729404556",
      "user_id": "xyz",
      "doc_type": "packaging_specification",
      "year": "2023",
      "filename": "pharma-blob-sample.pdf",
      "page_number": 3,
      "text": "Packaging Component Specification Document Number: PKG-SPEC-2023-0847 Revision: C Effective Date: 2023-11-08 Component: AKTA ready Flow Kit Packaging Assembly Drawing Reference: DWG-29477427-PKG-R3 1....",
      "match_score": 5
    },
    {
      "file_id": "06de3fe2-dd6b-5a9a-b0d0-57ca0b1f2987",
      "user_id": "xyz",
      "doc_type": "packaging_specification",
      "year": "2024",
      "filename": "pharma-blob-test.pdf",
      "page_number": 4,
      "text": "West Pharmaceutical Services, Inc. 347 Oliver Street, Jersey Shore, PA 17740 Tel: +1 (570) 398-5200 | www.westpharm

In [ ]:
with open("routing_result.json", "w") as f:
    json.dump(result, f, indent=2)
print("wrote routing_result.json")

wrote routing_result.json


### A batch of queries

Ten questions with the document type each one *should* land on, including two deliberately
vague ones, so the failure mode is measured rather than assumed.

In [ ]:
EVAL = [
    ("What are the part numbers listed in this document?",              "certificate_of_quality"),
    ("What was the Shore A hardness result for each batch?",            "certificate_of_quality"),
    ("Does this product contain any animal derived material?",          "bse_tse_declaration"),
    ("What blister tray material is used in the packaging?",            "packaging_specification"),
    ("Can the pump tubing be autoclaved?",                              "material_description"),
    ("When was the supplier last audited and what was the result?",     "supplier_qualification"),
    ("Where were the assemblies manufactured and how are they traced?", "chain_of_custody"),
    ("Who sent this package and which documents are included?",         "cover_letter"),
    ("Tell me about this package",                                      "unknown"),
    ("Is everything in order?",                                         "unknown"),
]

rows = []
for q, expected in EVAL:
    r = route_query(q, top_k=2)
    top = r["matched_documents"][0] if r["matched_documents"] else {}
    rows.append({
        "query": q[:46],
        "expected": expected,
        "predicted": r["predicted_doc_type"],
        "ok": r["predicted_doc_type"] == expected,
        "widened": r["widened_to_full_store"],
        "top_hit": f"{top.get('filename','-')[:14]} p{top.get('page_number','-')}",
        "score": top.get("match_score", 0),
    })

eval_df = pd.DataFrame(rows)
print(f"routing accuracy: {eval_df['ok'].sum()}/{len(eval_df)}\n")
print(eval_df.to_string(index=False))
eval_df

[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

routing accuracy: 8/10

                                         query                expected               predicted    ok  widened           top_hit  score
What are the part numbers listed in this docum  certificate_of_quality packaging_specification False    False pharma-blob-sa p3      5
What was the Shore A hardness result for each   certificate_of_quality  certificate_of_quality  True    False pharma-blob-te p2      7
Does this product contain any animal derived m     bse_tse_declaration     bse_tse_declaration  True    False pharma-blob-te p5     19
What blister tray material is used in the pack packaging_specification packaging_specification  True    False pharma-blob-sa p3     11
            Can the pump tubing be autoclaved?    material_description    material_description  True    False pharma-blob-sa p6      5
When was the supplier last audited and what wa  supplier_qualification  supplier_qualification  True    False pharma-blob-sa p7      8
Where were the assemblies manuf

,query,expected,predicted,ok,widened,top_hit,score
0,What are the part numbers listed in this d...,certificate_of_quality,packaging_specification,False,False,pharma-blob-sa p3,5
1,What was the Shore A hardness result for e...,certificate_of_quality,certificate_of_quality,True,False,pharma-blob-te p2,7
2,Does this product contain any animal deriv...,bse_tse_declaration,bse_tse_declaration,True,False,pharma-blob-te p5,19
3,What blister tray material is used in the ...,packaging_specification,packaging_specification,True,False,pharma-blob-sa p3,11
4,Can the pump tubing be autoclaved?,material_description,material_description,True,False,pharma-blob-sa p6,5
5,When was the supplier last audited and wha...,supplier_qualification,supplier_qualification,True,False,pharma-blob-sa p7,8
6,Where were the assemblies manufactured and...,chain_of_custody,chain_of_custody,True,False,pharma-blob-sa p9,3
7,Who sent this package and which documents ...,cover_letter,cover_letter,True,False,pharma-blob-te p0,6
8,Tell me about this package,unknown,packaging_specification,False,False,pharma-blob-sa p3,0
9,Is everything in order?,unknown,unknown,True,True,pharma-blob-sa p9,1


In [ ]:
# Every raw generation, so a wrong route can be traced to what the model actually said
print(len(RAW_OUTPUTS), "generations")
for r in RAW_OUTPUTS[-10:]:
    print(repr(r[:90]))

32 generations
'{"doc_type": "packaging_specification"}'
'{"doc_type": "certificate_of_quality"}'
'{"doc_type": "bse_tse_declaration"}'
'{"doc_type": "packaging_specification"}'
'{"doc_type": "material_description"}'
'{"doc_type": "supplier_qualification"}'
'{"doc_type": "chain_of_custody"}'
'{"doc_type": "cover_letter"}'
'{"doc_type": "packaging_specification"}'
'{"doc_type": "unknown"}'


## 10. What this run showed

**Numbers from this run:** 20 pages across 2 PDFs, 7 distinct `doc_type`s, routing accuracy
**8/10** on the evaluation set, 4 pages corrected by a deterministic override.

- **Metadata does the routing; the prompt only decides which bucket to open.** Once every
  page carries a `doc_type`, one dictionary filter cuts the candidate set from 20 pages to the
  2-5 that could hold the answer — no embeddings involved.
- **The closed label list is the load-bearing part of the prompt.** Both classifiers share
  it, so a query label and a page label are guaranteed comparable. Without that, filtering
  silently returns nothing.
- **The page classifier's weakness is letterheads.** Mistral labelled the Cytiva cover
  letter `packaging_specification` because the visible header is a company address block, not a
  document title; the same thing happened on the STERIS processing certificate and the
  compliance bulletin. Reading twelve header lines for an explicit document title and
  letting that beat the model fixed all four.
- **Vague questions are the failure mode, not obscure ones.** "Is everything in order?"
  correctly returns `unknown` and widens the store, but "Tell me about this package" is just
  specific enough to pull a confident wrong label out of the model.
- **"Part numbers" is genuinely ambiguous** and the one query the router still gets wrong:
  a released lot carries part numbers on the certificate, and a tray or carton carries them
  on the packaging spec. A single-label router has to pick one; the honest fix is to let
  the classifier return a ranked shortlist of types rather than a single winner.
- **Keyword scoring is a re-ranker, never the router.** It runs inside the filtered set;
  promoting it to the primary signal would put every page containing the word
  "specification" back in play.